# Uncapacitated Lot-Sizing (ULS)

The problem is to decide on a production plan for an $n$-period horizon for a single product. The basic model has the following data:

- $f_t$ is the fixed (setup) cost of producing in period $t$
- $p_t$ is the unit production cost in period $t$
- $h_t$ is the unit storage cost in period $t$
- $d_t$ is the demand in period $t$

## Decision Variables

- $x_t \geq 0$ is the amount produced in period $t$
- $s_t \geq 0$ is the stock at the end of period $t$ (with $s_0 = 0$)
- $y_t \in \{0, 1\}$ equals 1 if production occurs in period $t$, and 0 otherwise

To handle the fixed costs, we observe that a priori no upper bound is given on $x_t$. Thus we either must use a very large value $M$, or calculate an upper bound based on the problem data.

## Constraints

1. Inventory balance constraints:

$$
s_{t-1} + x_t = d_t + s_t \quad \text{for } t = 1, \ldots, n
$$

2. Production setup (variable upper bound) constraints:

$$
x_t \leq M y_t \quad \text{for } t = 1, \ldots, n
$$

where $M$ is a sufficiently large number, e.g. $M = \sum_{i=1}^{n} d_i$.

3. Variable domain constraints:

$$
s_0 = 0, \quad s_t, x_t \geq 0, \quad y_t \in \{0, 1\} \quad \text{for } t = 1, \ldots, n
$$

## Objective Function

Minimize the total cost (production, storage and fixed costs):

$$
\min \sum_{t=1}^{n} p_t x_t + \sum_{t=1}^{n} h_t s_t + \sum_{t=1}^{n} f_t y_t
$$

## Complete MIP Formulation

$$
\begin{aligned}
& \min \sum_{t=1}^{n} p_t x_t + \sum_{t=1}^{n} h_t s_t + \sum_{t=1}^{n} f_t y_t \\
& \text{subject to:} \\
& \quad s_{t-1} + x_t = d_t + s_t, \quad t = 1, \ldots, n \\
& \quad x_t \leq M y_t, \quad t = 1, \ldots, n \\
& \quad s_0 = 0 \\
& \quad s_t, x_t \geq 0, \quad t = 1, \ldots, n \\
& \quad y_t \in \{0, 1\}, \quad t = 1, \ldots, n
\end{aligned}
$$

## Tightened Formulation

If we impose that $s_n = 0$ (which loses nothing when $p_t, h_t \geq 0$, since producing more than the remaining demand never pays), then we can tighten the variable upper bound constraints to

$$
x_t \leq \left( \sum_{i=t}^{n} d_i \right) y_t \quad \text{for } t = 1, \ldots, n.
$$

A tighter big-$M$ gives a stronger LP relaxation, so the solver needs fewer branch-and-bound nodes.

Note also that by substituting

$$
s_t = \sum_{i=1}^{t} x_i - \sum_{i=1}^{t} d_i,
$$

the objective function can be rewritten as

$$
\sum_{t=1}^{n} c_t x_t + \sum_{t=1}^{n} f_t y_t - K,
$$

where

$$
c_t = p_t + h_t + h_{t+1} + \cdots + h_n
\qquad \text{and} \qquad
K = \sum_{t=1}^{n} h_t \left( \sum_{i=1}^{t} d_i \right).
$$


In [ ]:
!pip install -q gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import random

In [ ]:
def solve_uls(demand, prod_cost, hold_cost, fixed_cost, tighten=True, relax=False):
    T = len(demand)
    m = gp.Model("ULS")
    m.Params.OutputFlag = 0

    x = m.addVars(T, lb=0.0, name="x")
    s = m.addVars(T, lb=0.0, name="s")
    y = m.addVars(T, vtype=GRB.CONTINUOUS if relax else GRB.BINARY,
                  lb=0.0, ub=1.0, name="y")

    m.setObjective(
        gp.quicksum(prod_cost[t] * x[t] + hold_cost[t] * s[t] + fixed_cost[t] * y[t]
                    for t in range(T)),
        GRB.MINIMIZE,
    )

    for t in range(T):
        prev = s[t - 1] if t > 0 else 0
        m.addConstr(prev + x[t] == demand[t] + s[t], name=f"balance_{t}")
        M = sum(demand[t:]) if tighten else sum(demand)
        m.addConstr(x[t] <= M * y[t], name=f"setup_{t}")

    m.addConstr(s[T - 1] == 0, name="no_final_stock")

    m.optimize()
    return {
        "cost": m.ObjVal,
        "x": [x[t].X for t in range(T)],
        "s": [s[t].X for t in range(T)],
        "y": [round(y[t].X) for t in range(T)] if not relax else [y[t].X for t in range(T)],
    }


def wagner_whitin(demand, prod_cost, hold_cost, fixed_cost):
    """Exact DP: best[j] = min cost to satisfy demand of the first j periods."""
    T = len(demand)
    INF = float("inf")
    best = [0.0] + [INF] * T
    for j in range(1, T + 1):
        for i in range(1, j + 1):
            qty = sum(demand[i - 1:j])
            holding = sum(hold_cost[t - 1] * sum(demand[t:j]) for t in range(i, j))
            cost = fixed_cost[i - 1] + prod_cost[i - 1] * qty + holding
            best[j] = min(best[j], best[i - 1] + cost)
    return best[T]


def print_plan(res, periods=None):
    T = len(res["x"])
    periods = periods or list(range(1, T + 1))
    print("Production plan:")
    for t in range(T):
        setup = "setup" if res["y"][t] else "-"
        print(f"  Period {periods[t]}: produce {res['x'][t]:6.1f}   end stock {res['s'][t]:6.1f}   [{setup}]")
    print(f"Total cost: {res['cost']:.1f}")

Example 1: Four periods with setup costs. Because a setup is now expensive, the model batches production instead of producing every period.

In [ ]:
demand    = [20, 15, 25, 10]   # demand in each period
prod_cost = [3, 2, 4, 3]       # unit production cost
hold_cost = [1, 1, 1, 1]       # unit holding cost
fixed_cost = [60, 40, 50, 30]  # setup cost

res = solve_uls(demand, prod_cost, hold_cost, fixed_cost)
print_plan(res)

ww = wagner_whitin(demand, prod_cost, hold_cost, fixed_cost)
assert abs(res["cost"] - ww) < 1e-6
print(f"\nWagner-Whitin DP check passed: {ww:.1f}")

Restricted license - for non-production use only - expires 2027-11-29


Production plan:
  Period 1: produce   20.0   end stock    0.0   [setup]
  Period 2: produce   50.0   end stock   35.0   [setup]
  Period 3: produce    0.0   end stock   10.0   [-]
  Period 4: produce    0.0   end stock    0.0   [-]
Total cost: 305.0

Wagner-Whitin DP check passed: 305.0


Example 2: data stored in a dictionary

In [ ]:
data = {
    "periods":   [1, 2, 3],
    "demand":    {1: 12, 2: 18, 3: 10},
    "prod_cost": {1: 2, 2: 3, 3: 2},
    "hold_cost": {1: 1, 2: 1, 3: 1},
    "fixed_cost": {1: 30, 2: 20, 3: 30},
}
P = data["periods"]
d  = [data["demand"][t]    for t in P]
p  = [data["prod_cost"][t] for t in P]
h  = [data["hold_cost"][t] for t in P]
f  = [data["fixed_cost"][t] for t in P]

res = solve_uls(d, p, h, f)
print_plan(res, periods=P)

ww = wagner_whitin(d, p, h, f)
assert abs(res["cost"] - ww) < 1e-6
print(f"\nWagner-Whitin DP check passed: {ww:.1f}")

Production plan:
  Period 1: produce   40.0   end stock   28.0   [setup]
  Period 2: produce    0.0   end stock   10.0   [-]
  Period 3: produce    0.0   end stock    0.0   [-]
Total cost: 148.0

Wagner-Whitin DP check passed: 148.0


Example 3:
The seed makes the instance reproducible. Demands, unit production costs and setup costs are random; holding cost is constant.

In [ ]:
random.seed(123)
T = 8
demand     = [random.randint(10, 30) for _ in range(T)]
prod_cost  = [random.randint(2, 5) for _ in range(T)]
hold_cost  = [1 for _ in range(T)]
fixed_cost = [random.randint(30, 80) for _ in range(T)]

print("Demands          :", demand)
print("Production costs :", prod_cost)
print("Setup costs      :", fixed_cost, "\n")

res = solve_uls(demand, prod_cost, hold_cost, fixed_cost)
print_plan(res)

ww = wagner_whitin(demand, prod_cost, hold_cost, fixed_cost)
assert abs(res["cost"] - ww) < 1e-6
print(f"\nWagner-Whitin DP check passed: {ww:.1f}")

Demands          : [11, 18, 12, 23, 18, 13, 11, 22]
Production costs : [4, 4, 2, 3, 3, 4, 4, 3]
Setup costs      : [40, 30, 57, 79, 35, 68, 54, 34] 

Production plan:
  Period 1: produce   29.0   end stock   18.0   [setup]
  Period 2: produce    0.0   end stock    0.0   [-]
  Period 3: produce   35.0   end stock   23.0   [setup]
  Period 4: produce    0.0   end stock    0.0   [-]
  Period 5: produce   42.0   end stock   24.0   [setup]
  Period 6: produce    0.0   end stock   11.0   [-]
  Period 7: produce    0.0   end stock    0.0   [-]
  Period 8: produce   22.0   end stock    0.0   [setup]
Total cost: 620.0

Wagner-Whitin DP check passed: 620.0


In [ ]:
opt        = solve_uls(demand, prod_cost, hold_cost, fixed_cost)["cost"]
lp_loose   = solve_uls(demand, prod_cost, hold_cost, fixed_cost, tighten=False, relax=True)["cost"]
lp_tight   = solve_uls(demand, prod_cost, hold_cost, fixed_cost, tighten=True,  relax=True)["cost"]

print(f"Optimal MIP cost               : {opt:.2f}")
print(f"LP bound, M = sum of all d     : {lp_loose:.2f}   (gap {100*(opt-lp_loose)/opt:.1f}%)")
print(f"LP bound, tightened bound      : {lp_tight:.2f}   (gap {100*(opt-lp_tight)/opt:.1f}%)")
assert lp_loose <= lp_tight + 1e-9 <= opt + 1e-9

Optimal MIP cost               : 620.00
LP bound, M = sum of all d     : 467.20   (gap 24.6%)
LP bound, tightened bound      : 521.17   (gap 15.9%)


In [ ]:
T = len(demand)
c = [prod_cost[t] + sum(hold_cost[t:]) for t in range(T)]                     # c_t = p_t + h_t + ... + h_n
K = sum(hold_cost[t] * sum(demand[:t + 1]) for t in range(T))                  # K = sum_t h_t * (d_1 + ... + d_t)
alt = sum(c[t] * res["x"][t] + fixed_cost[t] * res["y"][t] for t in range(T)) - K
print(f"Original objective : {res['cost']:.4f}")
print(f"Rewritten objective: {alt:.4f}")
assert abs(alt - res["cost"]) < 1e-6

Original objective : 620.0000
Rewritten objective: 620.0000
